# STEP 1: Setup and Configuration

In [ ]:
import re
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from openai import OpenAI
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

sys.path.insert(0, '..')
from utils.api_client import create_openrouter_client

client = create_openrouter_client()

total_tokens = 0
token_lock = threading.Lock()

In [ ]:
@dataclass
class Config:
    textbooks_dir: str = "../ocr_output"
    output_dir: str = "../data"
    
    problem_keywords: List[str] = field(default_factory=lambda: ['example', 'problem', 'exercise', 'homework'])
    solution_keywords: List[str] = field(default_factory=lambda: ['solution', 'answer', 'answers'])
    
    context_chars_before: int = 1000
    context_chars_after: int = 1000
    min_content_length: int = 100
    max_content_length: int = 20000
    
    model: str = "openai/gpt-5-mini"
    temperature: float = 0.1
    max_tokens: int = 10000
    max_workers: int = 25

config = Config()
Path(config.output_dir).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  Model: {config.model}")
print(f"  Workers: {config.max_workers}")
print(f"  Textbooks: {Path(config.textbooks_dir).absolute()}")
print(f"  Output: {Path(config.output_dir).absolute()}")

# STEP 2: Extract Problem-Solution Pairs

In [25]:
def extract_all_problem_blocks(text: str, filename: str, config: Config) -> List[Dict]:
    results = []
    
    problem_pattern = '|'.join(config.problem_keywords)
    problem_regex = rf'\b({problem_pattern})\s*(\d+(?:\.\d+)?)?'
    
    matches = list(re.finditer(problem_regex, text, re.IGNORECASE))
    print(f"  Found {len(matches)} potential problems in {filename}")
    
    for i, match in enumerate(matches):
        keyword = match.group(1)
        number = match.group(2) if match.group(2) else "unnumbered"
        match_pos = match.start()
        
        # Extract large chunk around this keyword
        start = max(0, match_pos - 500)
        
        # End at next problem keyword or max length
        if i < len(matches) - 1:
            end = min(match_pos + config.max_content_length, matches[i+1].start() + 200)
        else:
            end = min(match_pos + config.max_content_length, len(text))
        
        raw_content = text[start:end]
        raw_content = re.sub(r'  +', ' ', raw_content).strip()
        
        if len(raw_content) >= config.min_content_length:
            source = f"[{filename}] {keyword.title()} {number}"
            results.append({
                'source': source,
                'raw_content': raw_content
            })
    
    return results

def extract_from_all_files(config: Config) -> List[Dict]:
    textbooks_path = Path(config.textbooks_dir)
    all_blocks = []
    
    md_files = list(textbooks_path.glob("*.md"))
    print(f"Found {len(md_files)} markdown files\n")
    
    for md_file in tqdm(md_files, desc="Extracting problem blocks"):
        try:
            with open(md_file, 'r', encoding='utf-8') as f:
                text = f.read()
            
            filename = md_file.stem
            extracted = extract_all_problem_blocks(text, filename, config)
            all_blocks.extend(extracted)
            
        except Exception as e:
            print(f"Error processing {md_file.name}: {e}")
    
    print(f"\nExtracted {len(all_blocks)} problem blocks")
    return all_blocks

raw_blocks = extract_from_all_files(config)

with open(Path(config.output_dir) / "step1_raw_blocks.json", 'w', encoding='utf-8') as f:
    json.dump(raw_blocks, f, indent=2, ensure_ascii=False)

print(f"Saved to step1_raw_blocks.json")

Found 32 markdown files



Extracting problem blocks:   0%|          | 0/32 [00:00<?, ?it/s]

  Found 317 potential problems in file_1
  Found 55 potential problems in file_10
  Found 333 potential problems in file_11
  Found 640 potential problems in file_12
  Found 799 potential problems in file_13
  Found 630 potential problems in file_14
  Found 719 potential problems in file_15
  Found 144 potential problems in file_16
  Found 464 potential problems in file_17
  Found 232 potential problems in file_18
  Found 335 potential problems in file_19
  Found 608 potential problems in file_2
  Found 167 potential problems in file_20
  Found 1296 potential problems in file_21
  Found 149 potential problems in file_22
  Found 234 potential problems in file_23
  Found 249 potential problems in file_24
  Found 162 potential problems in file_25
  Found 217 potential problems in file_26
  Found 96 potential problems in file_27
  Found 313 potential problems in file_28
  Found 114 potential problems in file_29
  Found 433 potential problems in file_3
  Found 343 potential problems in file

# STEP 3: Quality Validation

In [26]:
# STEP 3: LLM Filtering - STRICT validation

PROBLEM_SOLUTION_FILTER_PROMPT = """You are filtering textbook content to identify COMPLETE problem-solution pairs.

A block is VALID only if ALL of these are true:
1. Contains an ACTUAL problem/question being asked (not just a reference like "see Example 2.4")
2. Contains the SOLUTION/ANSWER to that problem in the same text
3. The solution actually solves the problem stated (not a different problem)
4. Is complete - not cut off mid-solution
5. Not just a problem statement for homework (must have the answer included)

A block is INVALID if ANY of these are true:
- Just mentions a problem number without the actual problem (e.g., "as shown in Example 3")
- Problem statement exists but NO solution/answer is provided
- Solution is for a different problem than the one stated
- Text is cut off or incomplete
- Just a chapter heading or section reference

Content to evaluate:
{content}

Think step by step:
1. Is there an actual problem being asked? (what specifically is being asked?)
2. Is there a solution/answer in this text? (where does it start?)
3. Does the solution address the problem stated?
4. Is it complete?

Respond with ONLY: VALID or INVALID"""

def filter_single_block(item: Dict, config: Config) -> Tuple[Dict, bool]:
    prompt = PROBLEM_SOLUTION_FILTER_PROMPT.format(content=item['raw_content'])
    
    try:
        response = client.chat.completions.create(
            model=config.model,
            max_tokens=100,
            temperature=config.temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        
        global total_tokens
        with token_lock:
            total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        
        text = response.choices[0].message.content.strip().upper()
        is_valid = "VALID" in text and "INVALID" not in text
        return item, is_valid
        
    except Exception as e:
        return item, False

def filter_all_parallel(raw_blocks: List[Dict], config: Config) -> List[Dict]:
    filtered = []
    
    print(f"Filtering with LLM using {config.max_workers} workers...")
    
    with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        future_to_item = {
            executor.submit(filter_single_block, item, config): item 
            for item in raw_blocks
        }
        
        for future in tqdm(as_completed(future_to_item), total=len(raw_blocks), desc="Filtering"):
            try:
                item, is_valid = future.result()
                if is_valid:
                    filtered.append(item)
            except Exception:
                pass
    
    print(f"\nFiltering complete: {len(filtered)} / {len(raw_blocks)} valid ({len(filtered)/len(raw_blocks)*100:.1f}%)")
    print(f"Tokens used: {total_tokens:,}")
    
    return filtered

validated_pairs = filter_all_parallel(raw_blocks, config)

with open(Path(config.output_dir) / "step2_validated_pairs.json", 'w', encoding='utf-8') as f:
    json.dump(validated_pairs, f, indent=2, ensure_ascii=False)

print(f"Saved to step2_validated_pairs.json")

Filtering with LLM using 10 workers...


Filtering:   0%|          | 0/13200 [00:00<?, ?it/s]


Filtering complete: 1568 / 13200 valid (11.9%)
Tokens used: 13,484,351
Saved to step2_validated_pairs.json


# STEP 4: Make Self-Contained + Format as Q/R/A

In [ ]:
COMBINED_QRA_PROMPT = """You are reformatting textbook problems into structured Question-Reasoning-Answer format.

Your task:
1. Make the question SELF-CONTAINED by integrating any referenced content
2. Extract the reasoning/solution methodology
3. Format the final answer according to STRICT rules

QUESTION Requirements:
- Include the complete problem statement
- If the problem references external content (e.g., "use equation 1.1", "from the data above", "as shown in the figure"), find that content in the surrounding text and integrate it DIRECTLY into the question
- Make it so someone reading ONLY the question can solve it without seeing any other text
- For multi-part questions (a, b, c, etc.), include ALL parts

REASONING Requirements:
- Include ALL solution steps, calculations, and logical reasoning
- Keep formulas, equations, and intermediate calculations
- Maintain the step-by-step solution process
- For multi-part questions, include reasoning for ALL parts

ANSWER Requirements - STRICT FORMATTING:
The answer must follow these rules EXACTLY for easy parsing:

For SINGLE answers, use ONE of these formats:
- Integer: 42
- Decimal: 3.14
- Fraction (reduced): 1/2
- Expression: 2x + 3
- Formula: P(A) = 0.5
- Boolean: True OR False
- Text (only if unavoidable): brief phrase

For MULTIPLE answers (multi-part questions):
- Separate with " | " (space-pipe-space)
- Example: 0.85 | 1200 | True
- Each part follows the single answer rules above

CRITICAL RULES:
- NO explanatory text in the answer
- NO units unless essential for understanding (prefer just the number)
- NO sentences like "The answer is..." or "Therefore..."
- NO extra punctuation
- For percentages: use decimal form (0.85 not 85%)
- For probabilities: use decimal form (0.5 not 50%)
- ALWAYS reduce fractions to simplest form

Content to reformat:
{content}

Output format:
QUESTION:
[complete self-contained question with all referenced data/equations integrated]

REASONING:
[step-by-step solution with all calculations]

ANSWER:
[final answer only, following strict formatting rules above]"""

def format_qra_single(item: Dict, config: Config) -> Tuple[Dict, Optional[Dict]]:
    prompt = COMBINED_QRA_PROMPT.format(content=item['raw_content'])
    
    try:
        response = client.chat.completions.create(
            model=config.model,
            max_tokens=config.max_tokens,
            temperature=config.temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        
        global total_tokens
        with token_lock:
            total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        
        text = response.choices[0].message.content
        
        question_match = re.search(r'QUESTION:\s*\n(.+?)(?=\n\s*REASONING:)', text, re.DOTALL)
        reasoning_match = re.search(r'REASONING:\s*\n(.+?)(?=\n\s*ANSWER:)', text, re.DOTALL)
        answer_match = re.search(r'ANSWER:\s*\n(.+?)$', text, re.DOTALL)
        
        if question_match and reasoning_match and answer_match:
            return item, {
                'source': item['source'],
                'question': question_match.group(1).strip(),
                'reasoning': reasoning_match.group(1).strip(),
                'answer': answer_match.group(1).strip()
            }
        
        return item, None
        
    except Exception:
        return item, None

def format_all_qra(validated_items: List[Dict], config: Config) -> List[Dict]:
    formatted = []
    
    print(f"Formatting as Q/R/A with {config.max_workers} workers...")
    
    with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        future_to_item = {
            executor.submit(format_qra_single, item, config): item 
            for item in validated_items
        }
        
        for future in tqdm(as_completed(future_to_item), total=len(validated_items), desc="Formatting"):
            try:
                item, result = future.result()
                if result:
                    formatted.append(result)
            except Exception:
                pass
    
    print(f"\nFormatted: {len(formatted)} / {len(validated_items)} successful ({len(formatted)/len(validated_items)*100:.1f}%)")
    print(f"Tokens used: {total_tokens:,}")
    
    return formatted

<>:36: SyntaxWarning: invalid escape sequence '\c'
<>:36: SyntaxWarning: invalid escape sequence '\c'
C:\Users\alexa\AppData\Local\Temp\ipykernel_21712\280246294.py:36: SyntaxWarning: invalid escape sequence '\c'
  - Formula: P(A) = 0.5, P(A) = P(C \cap D)


In [7]:
# Load validated pairs from previous step
with open(Path(config.output_dir) / "step2_validated_pairs.json", 'r', encoding='utf-8') as f:
    validated_pairs = json.load(f)

print(f"Loaded {len(validated_pairs)} validated pairs")

final_dataset = format_all_qra(validated_pairs, config)

with open(Path(config.output_dir) / "step3_qra_formatted.json", 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"Saved to step3_qra_formatted.json")

Loaded 1568 validated pairs
Formatting as Q/R/A with 25 workers...


Formatting:   0%|          | 0/1568 [00:00<?, ?it/s]


Formatted: 0 / 1568 successful (0.0%)
Tokens used: 13,447,037
Saved to step3_qra_formatted.json


# STEP 5: Quality Scoring

In [ ]:
QUALITY_SCORING_PROMPT = """You are scoring the quality of a reasoning trajectory based on 7 dimensions.

Question: {question}
Reasoning: {reasoning}
Answer: {answer}

Score each dimension as 0 (fails) or 1 (passes):

1. Internal Consistency: Does the reasoning logically lead to the answer?
2. Term Overlap: Are key terms from the question addressed in reasoning?
3. Reasoning Steps: Are there clear, identifiable steps (at least 2-3)?
4. Logical Coherence: Does each step follow from the previous?
5. Content Diversity: Does it show multiple concepts/approaches when appropriate?
6. Task Relevance: Does it directly address what the question asks?
7. Instruction Alignment: Does it follow a logical problem-solving structure?

Output format (one score per line):
1: [0 or 1]
2: [0 or 1]
3: [0 or 1]
4: [0 or 1]
5: [0 or 1]
6: [0 or 1]
7: [0 or 1]
TOTAL: [sum]"""

def score_quality_single(item: Dict, config: Config) -> Tuple[Dict, int]:
    prompt = QUALITY_SCORING_PROMPT.format(
        question=item['question'],
        reasoning=item['reasoning'],
        answer=item['answer']
    )
    
    try:
        response = client.chat.completions.create(
            model=config.model,
            max_tokens=200,
            temperature=config.temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        
        global total_tokens
        with token_lock:
            total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        
        text = response.choices[0].message.content
        total_match = re.search(r'TOTAL:\s*(\d+)', text)
        
        if total_match:
            score = int(total_match.group(1))
            return item, score
        
        return item, 0
        
    except Exception:
        return item, 0

def score_all_quality(dataset: List[Dict], config: Config, min_score: int = 5) -> List[Dict]:
    scored = []
    
    print(f"Scoring quality with {config.max_workers} workers...")
    
    with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        future_to_item = {
            executor.submit(score_quality_single, item, config): item 
            for item in dataset
        }
        
        for future in tqdm(as_completed(future_to_item), total=len(dataset), desc="Scoring"):
            try:
                item, score = future.result()
                if score >= min_score:
                    item['quality_score'] = score
                    scored.append(item)
            except Exception:
                pass
    
    print(f"\nScoring complete: {len(scored)} / {len(dataset)} passed (>={min_score}/7) ({len(scored)/len(dataset)*100:.1f}%)")
    print(f"Total tokens used: {total_tokens:,}")
    
    return scored

In [ ]:
high_quality_dataset = score_all_quality(final_dataset, config, min_score=5)

with open(Path(config.output_dir) / "step5_high_quality_final.json", 'w', encoding='utf-8') as f:
    json.dump(high_quality_dataset, f, indent=2, ensure_ascii=False)

print(f"Saved to step4_high_quality_final.json")

# STEP 6: Export Formats

In [ ]:
import pandas as pd

if high_quality_dataset:
    df = pd.DataFrame(high_quality_dataset)
    
    #df.to_csv(Path(config.output_dir) / "final_dataset.csv", index=False)
    #print("Exported CSV")
    
    with open(Path(config.output_dir) / "final_dataset.json", 'w', encoding='utf-8') as f:
        json.dump(high_quality_dataset, f, indent=2, ensure_ascii=False)
    print("Exported JSON")
    
    finetuning_data = []
    for item in high_quality_dataset:
        finetuning_data.append({
            "messages": [
                {
                    "role": "user",
                    "content": f"Question: {item['question']}\n\nProvide detailed reasoning and the answer."
                },
                {
                    "role": "assistant",
                    "content": f"Reasoning:\n{item['reasoning']}\n\nAnswer: {item['answer']}"
                }
            ]
        })
    
    with open(Path(config.output_dir) / "finetuning_format.jsonl", 'w', encoding='utf-8') as f:
        for item in finetuning_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print("Exported fine-tuning format")
    
    print(f"\nFinal dataset size: {len(high_quality_dataset)}")
    print(f"Average quality score: {df['quality_score'].mean():.2f}/7")
    print(f"\nAll files saved to: {Path(config.output_dir).absolute()}")